# Text Cleaning and Citation Metadata

This notebook cleans extracted research-paper text and preserves page-level metadata for later citation.

In [3]:
import json
import re
import unicodedata
from pathlib import Path

In [4]:
candidate_paths = [
    Path("../data/curated_papers_text.json").resolve(),
    Path(r"C:/Users/brant/OneDrive/Documents/Projects/Zayi-1B-agentic-internal-knowledge-system-for-r-and-d/data/curated_papers_text.json")
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    searched_paths = "\n".join(f"- {path}" for path in candidate_paths)
    raise FileNotFoundError(
        "Dataset not found. Open the local project folder in VS Code, then rerun this cell."
        f"\nSearched paths:\n{searched_paths}"
    )

output_path = data_path.with_name("cleaned_papers_text.json")

with data_path.open("r", encoding="utf-8") as file:
    data = json.load(file)

print(f"Loaded {len(data)} papers from {data_path}")

Loaded 10 papers from C:\Users\brant\OneDrive\Documents\Projects\Zayi-1B-agentic-internal-knowledge-system-for-r-and-d\data\curated_papers_text.json


In [5]:
def clean_extracted_text(text):
    text = unicodedata.normalize("NFKC", text or "")
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", " ", text)
    text = re.sub(r"(?<=\w)-\s*\n\s*(?=\w)", "", text)
    text = re.sub(r"[ \t]*\n[ \t]*", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def infer_section(text):
    section_patterns = [
        r"\b(?:abstract|introduction|background|related work|methodology|methods|experiments|results|discussion|conclusion|limitations|acknowledgements?|references)\b",
        r"\b(?:appendix|appendices)\s+[A-Z]\b",
        r"\b\d+(?:\.\d+)*\s+[A-Z][A-Za-z][^.!?]{0,80}"
    ]

    for pattern in section_patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            return match.group(0).strip()

    return "Unspecified"

In [6]:
cleaned_records = []

for paper in data:
    for page in paper.get("pages", []):
        raw_text = page.get("text", "")
        cleaned_text = clean_extracted_text(raw_text)
        page_number = page["page_number"]

        cleaned_records.append({
            "paper_id": paper["id"],
            "title": paper["title"],
            "category": paper["category"],
            "section": infer_section(raw_text),
            "page_number": page_number,
            "source_url": paper["pdf_url"],
            "citation_key": f"{paper['id']}-p{page_number}",
            "cleaned_text": cleaned_text,
            "text_length": len(cleaned_text),
            "word_count": len(cleaned_text.split()),
            "is_empty": not bool(cleaned_text)
        })

print(f"Built {len(cleaned_records)} page records.")

Built 218 page records.


In [7]:
assert len(cleaned_records) == sum(paper["total_pages"] for paper in data)
assert all(record["cleaned_text"] for record in cleaned_records)
assert len({record["citation_key"] for record in cleaned_records}) == len(cleaned_records)
assert all(record["source_url"] for record in cleaned_records)
assert not any(
    re.search(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]|\n|\t", record["cleaned_text"])
    for record in cleaned_records
)

print("Validation passed.")
print(json.dumps(cleaned_records[0], indent=2, ensure_ascii=False))

Validation passed.
{
  "paper_id": "2005.11401",
  "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
  "category": "RAG",
  "section": "Abstract",
  "page_number": 1,
  "source_url": "https://arxiv.org/pdf/2005.11401.pdf",
  "citation_key": "2005.11401-p1",
  "cleaned_text": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks Patrick Lewis†‡, Ethan Perez⋆, Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†, Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela† †Facebook AI Research;‡University College London;⋆New York University; plewis@fb.com Abstract Large pre-trained language models have been shown to store factual knowledge in their parameters, and achieve state-of-the-art results when fine-tuned on downstream NLP tasks. However, their ability to access and precisely manipulate knowledge is still limited, and hence on knowledge-intensive tasks, their performance lags behind task-

In [8]:
with output_path.open("w", encoding="utf-8") as file:
    json.dump(cleaned_records, file, ensure_ascii=False, indent=2)

print(f"Saved cleaned records to {output_path}")

Saved cleaned records to C:\Users\brant\OneDrive\Documents\Projects\Zayi-1B-agentic-internal-knowledge-system-for-r-and-d\data\cleaned_papers_text.json


## Output

The cleaned page-level records are saved to `data/cleaned_papers_text.json`.

Each record includes the paper title, inferred section, page number, source URL, and a citation key such as `2005.11401-p1`.